In [1]:
import shutil

src = "/kaggle/input/datasets/khushvaghela/public-repo/kaggle-cell-tracking-competition-main"
dst = "/kaggle/working/kaggle-cell-tracking-competition-main"

shutil.copytree(src, dst)

'/kaggle/working/kaggle-cell-tracking-competition-main'

In [2]:
import os
import subprocess
import sys

wheel_dir = "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels"

if not os.path.exists(wheel_dir):
    print(f"Error: Path {wheel_dir} not found!")
else:
    # Get all wheels
    all_wheels = [w for w in os.listdir(wheel_dir) if w.endswith(".whl")]
    
    # DANGEROUS packages that break Kaggle if overwritten from offline wheels
    skip_prefixes = ("numpy", "scipy", "pandas", "torch", "nvidia", "polars", "pyarrow")
    
    safe_wheels = []
    for w in all_wheels:
        # If the wheel name does NOT start with any of the dangerous prefixes, keep it
        if not any(w.lower().startswith(prefix) for prefix in skip_prefixes):
            safe_wheels.append(os.path.join(wheel_dir, w))
            
    print(f"Found {len(safe_wheels)} safe wheels to install (skipping {len(all_wheels) - len(safe_wheels)} dangerous ones).")
    
    # Run the pip install command with only the safe wheels
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"] + safe_wheels
    
    print("Installing...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("Success! All safe packages installed.")
    else:
        print("Installation failed:")
        print(result.stderr)

Found 57 safe wheels to install (skipping 5 dangerous ones).
Installing...
Success! All safe packages installed.


In [3]:
import sys
import os
import torch
import pandas as pd
import polars as pl

# 1. ADD YOUR CUSTOM FILES TO THE PYTHON PATH
CODE_DATASET_PATH = "/kaggle/input/datasets/khushvaghela/train-u-net/Train U-NET"
if CODE_DATASET_PATH not in sys.path:
    sys.path.append(CODE_DATASET_PATH)

# 2. Apply Polars patch
if not hasattr(pl, 'Float16'):
    pl.Float16 = pl.Float32
    print("Patched missing pl.Float16")

# 3. Import tracksdata
import tracksdata
print("Tracksdata imported successfully!")

Patched missing pl.Float16
Tracksdata imported successfully!


/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


In [4]:
"""
End-to-End Pipeline (Tracksdata Native).

Executes Phase 1 (Detection) -> Phase 2/3/4 (Native Graph + ILP Solve) -> 
Phase 4.5 (Division Validation) -> Submission CSV formatting.
"""

from __future__ import annotations

import os
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import torch

import data_io as bio_io
import preprocessing as prep
import detect
import tracksdata_pipeline as tdp
import division_validation as dv

# Import model architectures
from detector_model import Detector3D, DetectorConfig
from learned_association import EdgeScorerMLP
from division_mlp import DivisionScorerMLP
from pipeline_config import DOWNSAMPLE

# Standard Kaggle submission columns[cite: 10]
COLUMNS: tuple[str, ...] = (
    "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id",
)


def load_models(
    detector_path: str, 
    division_scorer_path: str, 
    edge_scorer_path: str | None = None,
    device: str | torch.device = "cpu"
) -> tuple[Detector3D, DivisionScorerMLP, EdgeScorerMLP | None]:
    """Loads all trained models from their Kaggle dataset checkpoint paths."""
    
    # 1. Load Phase 1 Detector
    torch.serialization.add_safe_globals([DetectorConfig])
    det_checkpoint = torch.load(detector_path, map_location=device, weights_only=False)
    detector = Detector3D(det_checkpoint["config"]).to(device)
    detector.load_state_dict(det_checkpoint["model_state_dict"])
    detector.eval()

    # 2. Load Phase 4.5 Division Scorer[cite: 9]
    div_scorer = DivisionScorerMLP()
    div_scorer.load_state_dict(torch.load(division_scorer_path, map_location=device, weights_only=True))
    div_scorer.eval()

    # 3. Load Phase 2 Edge Scorer (Optional)
    edge_scorer = None
    if edge_scorer_path is not None and os.path.exists(edge_scorer_path):
        edge_scorer = EdgeScorerMLP()
        edge_scorer.load_state_dict(torch.load(edge_scorer_path, map_location=device, weights_only=True))
        edge_scorer.eval()

    return detector, div_scorer, edge_scorer

def run_sample_end_to_end(
    detector: Detector3D,
    div_scorer: DivisionScorerMLP,
    sample: bio_io.Sample,
    edge_scorer: EdgeScorerMLP | None = None,
    downsample: tuple[int, int, int] = DOWNSAMPLE,
    detect_threshold: float = 0.30,
    nms_min_distance_um: float = 2.5,   # <-- UPDATED for SIGMA=2.0 U-Net
    division_threshold: float = 0.85,   # <-- UPDATED from your best Jaccard score
    use_ilp: bool = False,              # <-- UPDATED: Default to the robust NN solver
) -> dv.TrackingResult:
    """Runs the complete tracksdata native pipeline on a single sample."""
    
    device = next(detector.parameters()).device
    vol = bio_io.open_zarr_volume(sample.zarr_path)
    n_frames = vol.shape[0]

    detections_per_frame: dict[int, pd.DataFrame] = {}
    raw_frames_by_t: dict[int, np.ndarray] = {}

    print(f"--- Phase 1: Detecting cells across {n_frames} frames ---")
    for t in range(n_frames):
        raw_frame = np.asarray(vol[t])[None]
        raw_frames_by_t[t] = raw_frame
        
        frame_processed = prep.preprocess_window(raw_frame, downsample)[0]
        
        heatmap = detect.compute_heatmap_single_frame(detector, frame_processed)
        scale_down = prep.downsampled_scale(bio_io.NATIVE_SCALE, downsample)
        peaks_down = detect.extract_peaks_single_frame(heatmap, detect_threshold, nms_min_distance_um, scale_down)
        detections_per_frame[t] = detect.peaks_to_native_coords(peaks_down, downsample)

    print("--- Phase 2 & 3: Building multi-frame Candidate Graph ---")
    # <-- UPDATED: The exact config that safely captured the true divisions
    candidate_graph = tdp.build_candidate_graph(
        detections_per_frame, 
        bio_io.NATIVE_SCALE,
        config=tdp.CandidateGraphConfig(
            distance_threshold_um=25.0,  
            delta_t=1,                   
            n_neighbors=10, 
            neighbors_per_frame=True     
        ) 
    )

    print("--- Phase 4: Solving tracking network flow ---")
    if use_ilp:
        solution_graph = tdp.solve_ilp(candidate_graph, node_weight=-10.0)
    else:
        # <-- UPDATED: Using our proven champion
        solution_graph = tdp.solve_nearest_neighbors(candidate_graph, max_children=2)

    tracking_result = tdp.graph_view_to_tracking_result(solution_graph)

    # <-- CRITICAL ADDITION: Convert back to native voxels for validation and submission
    scale_arr = np.array(bio_io.NATIVE_SCALE)
    tracking_result.nodes["z"] /= scale_arr[0]
    tracking_result.nodes["y"] /= scale_arr[1]
    tracking_result.nodes["x"] /= scale_arr[2]

    print("--- Phase 4.5: Validating solver divisions ---")
    final_result = dv.validate_divisions(
        tracking_result,
        div_scorer,
        vol,              # <--- THE FIX: Pass the actual Zarr volume!
        bio_io.NATIVE_SCALE,
        threshold=division_threshold
    )
    
    return final_result

def tracking_result_to_submission_rows(tracking_result: dv.TrackingResult, dataset_name: str) -> pl.DataFrame:
    """
    Converts a TrackingResult into the exact Kaggle submission schema[cite: 10].
    Unused per-row-type fields are set to -1, and coordinates are rounded to integers[cite: 10].
    """
    nodes_df = tracking_result.nodes
    edges_df = tracking_result.edges

    # Format Nodes
    nodes_pl = pl.from_pandas(nodes_df).select(
        pl.lit(dataset_name).alias("dataset"),
        pl.lit("node").alias("row_type"),
        pl.col("node_id").cast(pl.Int64),
        pl.col("t").cast(pl.Int64),
        pl.col("z").cast(pl.Float64).round(0).cast(pl.Int64),
        pl.col("y").cast(pl.Float64).round(0).cast(pl.Int64),
        pl.col("x").cast(pl.Float64).round(0).cast(pl.Int64),
        pl.lit(-1, dtype=pl.Int64).alias("source_id"),
        pl.lit(-1, dtype=pl.Int64).alias("target_id"),
    )

    # Format Edges
    if len(edges_df) > 0:
        edges_pl = pl.from_pandas(edges_df).select(
            pl.lit(dataset_name).alias("dataset"),
            pl.lit("edge").alias("row_type"),
            pl.lit(-1, dtype=pl.Int64).alias("node_id"),
            pl.lit(-1, dtype=pl.Int64).alias("t"),
            pl.lit(-1, dtype=pl.Int64).alias("z"),
            pl.lit(-1, dtype=pl.Int64).alias("y"),
            pl.lit(-1, dtype=pl.Int64).alias("x"),
            pl.col("source_id").cast(pl.Int64),
            pl.col("target_id").cast(pl.Int64),
        )
    else:
        edges_pl = pl.DataFrame(schema=nodes_pl.schema)

    return pl.concat([nodes_pl, edges_pl])


def generate_final_submission(
    detector_path: str,
    division_scorer_path: str,
    test_dir: str,
    output_csv: str,
    device: str = "cuda",
) -> None:
    """Runs inference on all test samples and writes the final submission.csv."""
    
    print("Loading models...")
    detector, div_scorer, _ = load_models(detector_path, division_scorer_path, device=device)
    
    # Load all test samples (no ground truth needed)
    test_samples = bio_io.list_samples(test_dir)
    
    all_frames = []
    
    for sample in test_samples:
        print(f"\nProcessing sample: {sample.name}")
        tracking_result = run_sample_end_to_end(
            detector=detector,
            div_scorer=div_scorer,
            sample=sample,
            use_ilp=False  # <-- CRITICAL FIX: Turn off the ILP to unleash the NN + MLP combo!
        )
        
        # Flatten and format for this specific sample[cite: 10]
        sample_rows = tracking_result_to_submission_rows(tracking_result, sample.name)
        all_frames.append(sample_rows)
        print(f"Produced {sample_rows.height} submission rows for {sample.name}.")

    # Concatenate everything and add the 'id' index column[cite: 10]
    if all_frames:
        final_table = pl.concat(all_frames)
    else:
        final_table = pl.DataFrame(schema={c: (pl.String if c in ("dataset", "row_type") else pl.Int64) for c in COLUMNS})
        
    final_table = final_table.with_row_index("id")
    
    # Write to disk[cite: 10]
    output_path = Path(output_csv)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_table.write_csv(output_path)
    print(f"\nSuccess! Wrote {final_table.height} total rows to {output_path}")

In [5]:
# Example Kaggle Execution
TEST_DIRECTORY = bio_io.TEST_DIR
OUTPUT_FILE = "/kaggle/working/submission.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Where we will save the reconstructed file
repacked_pt_path = "/kaggle/working/detector_weights_best.pt"
unzipped_dir_path = "/kaggle/input/datasets/khushvaghela/best-weights/detector_weights_best/detector_weights_best"
print("Repacking unzipped PyTorch weights with correct 'archive/' structure...")

# 3. Create the ZIP file manually, forcing the 'archive/' parent folder
with zipfile.ZipFile(repacked_pt_path, 'w', zipfile.ZIP_STORED) as zipf:
    for root, dirs, files in os.walk(unzipped_dir_path):
        for file in files:
            file_path = os.path.join(root, file)
            # Calculate path relative to the extracted folder
            rel_path = os.path.relpath(file_path, unzipped_dir_path)
            
            # CRITICAL FIX: Prepend 'archive/' to the internal ZIP path
            zip_path = os.path.join('archive', rel_path)
            zipf.write(file_path, zip_path)

print(f"Successfully reconstructed weights at: {repacked_pt_path}")

DETECTOR_WEIGHTS = repacked_pt_path

# 2. Where we will save the reconstructed file
repacked_pt_path = "/kaggle/working/division_scorer_weights.pt"
unzipped_dir_path = "/kaggle/input/datasets/khushvaghela/best-weights/division_scorer_weights/division_scorer_weights"
print("Repacking unzipped PyTorch weights with correct 'archive/' structure...")

# 3. Create the ZIP file manually, forcing the 'archive/' parent folder
with zipfile.ZipFile(repacked_pt_path, 'w', zipfile.ZIP_STORED) as zipf:
    for root, dirs, files in os.walk(unzipped_dir_path):
        for file in files:
            file_path = os.path.join(root, file)
            # Calculate path relative to the extracted folder
            rel_path = os.path.relpath(file_path, unzipped_dir_path)
            
            # CRITICAL FIX: Prepend 'archive/' to the internal ZIP path
            zip_path = os.path.join('archive', rel_path)
            zipf.write(file_path, zip_path)

print(f"Successfully reconstructed weights at: {repacked_pt_path}")

DIVISION_WEIGHTS = repacked_pt_path

generate_final_submission(
    detector_path=DETECTOR_WEIGHTS,
    division_scorer_path=DIVISION_WEIGHTS,
    test_dir=TEST_DIRECTORY,
    output_csv=OUTPUT_FILE,
    device=device
)

Repacking unzipped PyTorch weights with correct 'archive/' structure...
Successfully reconstructed weights at: /kaggle/working/detector_weights_best.pt
Repacking unzipped PyTorch weights with correct 'archive/' structure...
Successfully reconstructed weights at: /kaggle/working/division_scorer_weights.pt
Loading models...

Processing sample: 44b6_0113de3b
--- Phase 1: Detecting cells across 100 frames ---
--- Phase 2 & 3: Building multi-frame Candidate Graph ---


Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

[08/16/26 05:25:48] WARNING  No nodes found for time point -1                                ]8;id=8683514;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=8683515;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#295\295]8;;\

                    WARNING  No valid edges found for time point 0                           ]8;id=8683521;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=8683522;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#337\337]8;;\

Adding edges: 100%|██████████| 100/100 [00:00<00:00, 227.00it/s]


...

Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

[08/16/26 05:34:36] WARNING  No nodes found for time point -1                                ]8;id=8683551;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=8683552;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#295\295]8;;\

                    WARNING  No valid edges found for time point 0                           ]8;id=8683557;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=8683558;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#337\337]8;;\

Adding edges: 100%|██████████| 100/100 [00:00<00:00, 102.89it/s]


--- Phase 4: Solving tracking network flow ---
--- Phase 4.5: Validating solver divisions ---
Pre-computing dictionaries for ultra-fast validation filtering...
Produced 136285 submission rows for 6bba_05db0fb1.

Success! Wrote 263180 total rows to /kaggle/working/submission.csv
